# nb_utils_ibge — Motores de Ingestão e Padronização (Fabric)

Este notebook contém as funções core para a ingestão de dados do IBGE (SIDRA) no Microsoft Fabric. Ele é projetado para ser escalável e suportar múltiplos clusters de municípios.

**Clusters Suportados:** Santos, Osasco e Mauá.

In [ ]:
import requests
import json
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, regexp_replace, trim, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# Workaround SSL para ambientes corporativos no Fabric
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
_orig_get = requests.get
requests.get = lambda *a, **kw: _orig_get(*a, **{**kw, "verify": False, "timeout": 30})

In [ ]:
# Configuração de Clusters
# Códigos verificados via IBGE Cidades (cidades.ibge.gov.br) + retorno real da API SIDRA
CLUSTERS = {
    "SANTOS": [3548500, 3551009, 3541000, 3518701, 3513504],
    # Santos · São Vicente · Praia Grande · Guarujá · Cubatão
    "OSASCO": [3534401, 3552809, 3552502, 3530607, 3556503, 3548807],
    # Osasco · Taboão da Serra · Suzano · Mogi das Cruzes · Várzea Paulista · São Caetano do Sul
    # 3552809 = Taboão da Serra  (era 3548708 = São Bernardo do Campo)
    # 3552502 = Suzano            (era 3552403 = Sumaré; antes era 3552205 = Sorocaba)
    # 3556503 = Várzea Paulista  (era 3556206 = Valinhos)
    # 3548807 = São Caetano do Sul (era 3549904 = São José dos Campos)
    "MAUA":   [3529401, 3510609, 3522505, 3543303],
    # Mauá · Carapicuíba · Itapevi · Ribeirão Pires
    # 3522505 = Itapevi           (era 3538709 = Piracicaba)
}

def get_all_municipios():
    all_codes = []
    for cluster in CLUSTERS.values():
        all_codes.extend(cluster)
    return sorted(list(set(all_codes)))

def get_municipios_by_cluster(cluster_name):
    if not cluster_name or cluster_name == "ALL":
        return get_all_municipios()
    return CLUSTERS.get(cluster_name.upper(), [])

In [ ]:
def fetch_sidra_fabric(table_code, variable, cluster_name=None, classifications=None):
    """
    Realiza a chamada para a API SIDRA e retorna um Spark DataFrame.
    """
    municipios = get_municipios_by_cluster(cluster_name)
    codes_str = ",".join(map(str, municipios))
    
    class_str = "" 
    if classifications:
        for c_id, c_val in classifications.items():
            class_str += f"/c{c_id}/{c_val}"
            
    url = f"https://apisidra.ibge.gov.br/values/t/{table_code}/n6/{codes_str}/v/{variable}/p/all{class_str}"
    
    print(f"[INFO] Buscando: {url}")
    response = requests.get(url)
    
    if response.status_code != 200:
        raise Exception(f"Erro na API SIDRA: {response.status_code} - {response.text}")
        
    data = response.json()
    if len(data) <= 1:
        print("[AVISO] Nenhum dado retornado pela API.")
        return None
        
    # Converte para Spark DataFrame (primeira linha é cabeçalho)
    df_pd = pd.DataFrame(data[1:], columns=data[0])
    return spark.createDataFrame(df_pd)

In [ ]:
def save_delta(df, table_name, mode="overwrite"):
    """
    Salva o DataFrame como uma tabela Delta no Lakehouse padrão.
    """
    if df is None:
        return
        
    print(f"[OK] Gravando tabela: {table_name} ({df.count()} registros)")
    df.write.format("delta").mode(mode).option("overwriteSchema", "true").saveAsTable(table_name)